In [ ]:
import os
import glob
import cv2
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, jaccard_score, precision_score, recall_score
import matplotlib.pyplot as plt

# -------------------------
# Configs
# -------------------------
IMG_SIZE = 512
NUM_CLASSES = 2
BATCH_SIZE = 4
NUM_EPOCHS = 50
LEARNING_RATE = 0.003
PATIENCE = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", DEVICE)

# -------------------------
# Dataset Class
# -------------------------
class LungDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].transpose(2, 0, 1)  # HWC → CHW
        img_tensor = torch.tensor(img, dtype=torch.float32) / 255.0
        mask_tensor = torch.tensor(self.masks[idx], dtype=torch.long)
        return img_tensor, mask_tensor

# -------------------------
# Load Data
# -------------------------
def load_data(image_dir, mask_dir):
    img_paths = sorted(glob.glob(os.path.join(image_dir, "*.png")))
    mask_paths = sorted(glob.glob(os.path.join(mask_dir, "*.png")))

    images = [cv2.resize(cv2.imread(p), (IMG_SIZE, IMG_SIZE)) for p in img_paths]
    masks = [cv2.resize(cv2.imread(p, 0), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST) for p in mask_paths]

    images = np.array(images)
    masks = np.array(masks)

    le = LabelEncoder()
    masks_flat = masks.reshape(-1, 1)
    masks_enc = le.fit_transform(masks_flat).reshape(masks.shape)

    return images, masks_enc

# -------------------------
# Model Components
# -------------------------
import torchvision.models as models

class ResNet50Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet50(pretrained=True)
        self.stage1 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.stage2 = base.layer1  # 256 channels
        self.stage3 = base.layer2  # 512 channels
        self.stage4 = base.layer3  # 1024 channels
        self.stage5 = base.layer4  # 2048 channels

    def forward(self, x):
        x = self.stage1(x)
        f1 = self.stage2(x)
        f2 = self.stage3(f1)
        f3 = self.stage4(f2)
        f4 = self.stage5(f3)
        return [f1, f2, f3, f4]

class MLPDecoder(nn.Module):
    def __init__(self, dims, embed_dim=256):
        super().__init__()
        self.proj = nn.ModuleList([nn.Conv2d(dim, embed_dim, 1) for dim in dims])
        self.conv = nn.Sequential(
            nn.Conv2d(embed_dim * len(dims), embed_dim, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim, embed_dim, 1)
        )

    def forward(self, features):
        target_size = features[0].shape[2:]  # spatial size (H, W)
        upsampled = []
        for i, feat in enumerate(features):
            feat = self.proj[i](feat)
            feat = F.interpolate(feat, size=target_size, mode='bilinear', align_corners=False)
            upsampled.append(feat)
        x = torch.cat(upsampled, dim=1)
        return self.conv(x)

class MLPBlock(nn.Module):
    def __init__(self, input_channels, hidden_channels):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv2d(input_channels, hidden_channels, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, input_channels, 1)
        )

    def forward(self, x):
        return self.mlp(x)

class GRUWrapper(nn.Module):
    def __init__(self, input_channels, hidden_size):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_channels,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )
        self.out_conv = nn.Conv2d(2 * hidden_size, input_channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B * H, W, C)  # (B*H, W, C)
        gru_out, _ = self.gru(x)
        gru_out = gru_out.reshape(B, H, W, -1).permute(0, 3, 1, 2)  # (B, 2H, H, W) → (B, 2H, H, W)
        return self.out_conv(gru_out)

class SegResNet(nn.Module):
    def __init__(self, num_classes=2, img_size=IMG_SIZE):
        super().__init__()
        self.img_size = img_size
        self.encoder = ResNet50Backbone()
        self.decoder = MLPDecoder([256, 512, 1024, 2048], embed_dim=256)

        self.mlp_block = MLPBlock(input_channels=256, hidden_channels=512)
        self.gru_block = GRUWrapper(input_channels=256, hidden_size=128)
        self.residual = nn.Conv2d(256, 256, 1)

        self.final_conv = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        features = self.encoder(x)
        x = self.decoder(features)

        identity = self.residual(x)
        x = self.mlp_block(x)
        x = self.gru_block(x)
        x = x + identity  # Residual connection

        x = self.final_conv(x)
        x = F.interpolate(x, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
        return x


# -------------------------
# Training, Evaluation, etc.
# -------------------------
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            out = model(x)
            preds = torch.argmax(out, dim=1).cpu().numpy()
            y_true.extend(y.numpy().flatten())
            y_pred.extend(preds.flatten())
    return y_true, y_pred

def plot_confusion_matrix(cm, classes, normalize=True, title='Confusion Matrix', cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm *= 100
    print(cm)
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45)
    plt.yticks(ticks, classes)
    thresh = cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

# -------------------------
# Main
# -------------------------
def main():
    base_path = r".../Cropped_Lung"
    train_img_dir = os.path.join(base_path, "Train/Image")
    train_mask_dir = os.path.join(base_path, "Train/Mask")
    test_img_dir = os.path.join(base_path, "Test/Image")
    test_mask_dir = os.path.join(base_path, "Test/Mask")

    print("Loading training data...")
    X_train_all, y_train_all = load_data(train_img_dir, train_mask_dir)
    X_train, X_val, y_train, y_val = train_test_split(X_train_all, y_train_all, test_size=0.1, random_state=42)

    train_loader = DataLoader(LungDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(LungDataset(X_val, y_val), batch_size=BATCH_SIZE)

    model = SegResNet(num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5, verbose=True)
    criterion = nn.CrossEntropyLoss()

    best_iou = 0
    epochs_no_improve = 0
    train_losses, val_losses, val_ious, train_ious = [], [], [], []

    for epoch in range(NUM_EPOCHS):
        train_loss = train(model, train_loader, optimizer, criterion)
        train_losses.append(train_loss)

        model.eval()
        total_val_loss = 0
        all_preds, all_targets = [], []
        train_preds, train_targets = [], []

        with torch.no_grad():
            for x, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                preds = torch.argmax(out, dim=1)
                train_preds.extend(preds.cpu().numpy().flatten())
                train_targets.extend(y.cpu().numpy().flatten())

            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                loss = criterion(out, y)
                total_val_loss += loss.item()
                preds = torch.argmax(out, dim=1)
                all_preds.extend(preds.cpu().numpy().flatten())
                all_targets.extend(y.cpu().numpy().flatten())

        val_loss = total_val_loss / len(val_loader)
        val_losses.append(val_loss)

        train_iou = jaccard_score(train_targets, train_preds, average='macro')
        val_iou = jaccard_score(all_targets, all_preds, average='macro')
        train_ious.append(train_iou)
        val_ious.append(val_iou)

        scheduler.step(val_iou)

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train IoU: {train_iou:.4f} | Val IoU: {val_iou:.4f}")

        # Early stopping
        if val_iou > best_iou:
            best_iou = val_iou
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_model.pth")
            print(">> Saved best model")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print("Early stopping triggered")
                break

    # Load best model for evaluation on test set
    print("Loading best model for testing...")
    model.load_state_dict(torch.load("best_model.pth"))

    print("Loading test data...")
    X_test, y_test = load_data(test_img_dir, test_mask_dir)
    test_loader = DataLoader(LungDataset(X_test, y_test), batch_size=BATCH_SIZE)

    y_true, y_pred = evaluate(model, test_loader)

    # Classification metrics
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    iou = jaccard_score(y_true, y_pred, average='macro')
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')

    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test F1 Score: {f1:.4f}")
    print(f"Test IoU: {iou:.4f}")
    print(f"Test Precision: {precision:.4f}")
    print(f"Test Recall: {recall:.4f}")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plot_confusion_matrix(cm, classes=['Background', 'Tumor'])

if __name__ == "__main__":
    main()
  